# 01 — Dependency-preserving pond data generator

## Goal

Create synthetic shrimp-pond time series when historical AQUASURE files are unavailable. This stage preserves dependence among **nine** water-quality variables.

**Used for:** supplying coherent multivariate sequences to the biological and forecasting notebooks.  
**Produces:** `farm_profiles.csv`, `pond_timeseries.csv`, and `copula_parameters.json`.


## Context & Methods

The reconstruction contains 12 farm profiles and 120-day production cycles. A Gaussian copula supplies correlated innovations; variable equations impose relationships such as lower dissolved oxygen under heat, organic load, and pathogen pressure.

### Key assumptions

- Synthetic data are development inputs, not observed Indonesian pond records.
- Real-time MVP: temperature, DO, pH, salinity/EC, and turbidity.
- Periodic auxiliary inputs: ammonia, nitrite, alkalinity, and pathogen pressure.
- Fixed seed: `20260830`.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 20260830
print(f"AQUASURE project root: {ROOT}")


## Data

### 1. Define nine variables and 12 farm profiles


In [ ]:
VARIABLES = [
    "temperature_c", "dissolved_oxygen_mg_l", "ph", "salinity_ppt",
    "ammonia_mg_l", "nitrite_mg_l", "turbidity_ntu",
    "alkalinity_mg_l", "pathogen_pressure",
]
REALTIME_MVP = ["temperature_c", "dissolved_oxygen_mg_l", "ph", "salinity_ppt", "turbidity_ntu"]
PERIODIC_AUXILIARY = ["ammonia_mg_l", "nitrite_mg_l", "alkalinity_mg_l", "pathogen_pressure"]

farms = pd.DataFrame({
    "farm_id": [f"FARM_{i:02d}" for i in range(1, 13)],
    "climate_zone": np.tile(["coastal_wet", "coastal_dry", "estuarine"], 4),
    "management_quality": np.linspace(0.35, 0.90, 12),
    "stocking_intensity": np.linspace(0.55, 1.15, 12)[::-1],
    "sum_insured_weight": np.linspace(0.75, 1.25, 12),
})
farms.to_csv(ARTIFACTS / "farm_profiles.csv", index=False)
print(farms.to_string(index=False))
print("\nForecast inputs (9):", VARIABLES)


### 2. Lock a valid copula dependence matrix


In [ ]:
correlation = np.array([
 [ 1.00,-0.55,-0.18, 0.08, 0.35, 0.30, 0.24,-0.08, 0.28],
 [-0.55, 1.00, 0.18,-0.04,-0.52,-0.45,-0.30, 0.14,-0.48],
 [-0.18, 0.18, 1.00, 0.10,-0.24,-0.18,-0.10, 0.42,-0.20],
 [ 0.08,-0.04, 0.10, 1.00, 0.06, 0.05, 0.08, 0.10, 0.04],
 [ 0.35,-0.52,-0.24, 0.06, 1.00, 0.58, 0.38,-0.20, 0.55],
 [ 0.30,-0.45,-0.18, 0.05, 0.58, 1.00, 0.32,-0.15, 0.49],
 [ 0.24,-0.30,-0.10, 0.08, 0.38, 0.32, 1.00,-0.08, 0.31],
 [-0.08, 0.14, 0.42, 0.10,-0.20,-0.15,-0.08, 1.00,-0.14],
 [ 0.28,-0.48,-0.20, 0.04, 0.55, 0.49, 0.31,-0.14, 1.00],
])
eigenvalue, eigenvector = np.linalg.eigh(correlation)
correlation = eigenvector @ np.diag(np.maximum(eigenvalue, 0.05)) @ eigenvector.T
scale = np.sqrt(np.diag(correlation))
correlation = correlation / np.outer(scale, scale)
np.linalg.cholesky(correlation)
print("Minimum eigenvalue:", np.linalg.eigvalsh(correlation).min().round(4))


### 3. Generate 120-day pond cycles


In [ ]:
N_CYCLES_PER_FARM, CYCLE_DAYS = 80, 120
records = []
rng = np.random.default_rng(SEED + 1)
for farm in farms.itertuples(index=False):
    for cycle_number in range(N_CYCLES_PER_FARM):
        innovation = rng.multivariate_normal(np.zeros(9), correlation, size=CYCLE_DAYS)
        day = np.arange(1, CYCLE_DAYS + 1)
        season = np.sin(2 * np.pi * (day + 17 * cycle_number) / 365.25)
        organic_load = (day / CYCLE_DAYS) * farm.stocking_intensity
        management = farm.management_quality
        temperature = 29.0 + 1.4 * season + 1.1 * innovation[:, 0]
        ammonia = np.clip(0.05 + 0.34 * organic_load - 0.17 * management + 0.10 * innovation[:, 4], 0, 1.5)
        nitrite = np.clip(0.03 + 0.22 * organic_load + 0.42 * ammonia + 0.07 * innovation[:, 5], 0, 1.2)
        pathogen = 1 / (1 + np.exp(-(-2.5 + 2.5 * organic_load + 1.1 * innovation[:, 8] - 1.2 * management)))
        dissolved_oxygen = np.clip(6.3 - 0.20 * (temperature - 28) - 1.45 * organic_load - 0.95 * ammonia + 0.48 * innovation[:, 1] + 0.65 * management, 1.2, 9.0)
        ph = np.clip(7.85 + 0.20 * innovation[:, 2] - 0.22 * ammonia, 6.6, 9.2)
        salinity = np.clip(19.0 + 5.0 * innovation[:, 3] + 2.0 * season, 3, 38)
        turbidity = np.clip(34 + 42 * organic_load + 18 * innovation[:, 6] + 16 * pathogen, 3, 180)
        alkalinity = np.clip(112 + 18 * innovation[:, 7] + 12 * management - 8 * organic_load, 55, 190)
        records.append(pd.DataFrame({
            "farm_id": farm.farm_id, "cycle_id": f"{farm.farm_id}_C{cycle_number+1:03d}", "day": day,
            "temperature_c": temperature, "dissolved_oxygen_mg_l": dissolved_oxygen,
            "ph": ph, "salinity_ppt": salinity, "ammonia_mg_l": ammonia,
            "nitrite_mg_l": nitrite, "turbidity_ntu": turbidity,
            "alkalinity_mg_l": alkalinity, "pathogen_pressure": pathogen,
        }))
pond = pd.concat(records, ignore_index=True)
pond.to_csv(ARTIFACTS / "pond_timeseries.csv", index=False)
params = {
    "variables": VARIABLES, "realtime_mvp": REALTIME_MVP, "periodic_auxiliary": PERIODIC_AUXILIARY,
    "farm_profiles": len(farms), "cycles_per_farm": N_CYCLES_PER_FARM,
    "cycle_days": CYCLE_DAYS, "seed": SEED, "correlation": correlation.round(6).tolist(),
}
(ARTIFACTS / "copula_parameters.json").write_text(json.dumps(params, indent=2))
print(f"Saved {len(pond):,} pond-day rows across {pond.cycle_id.nunique():,} cycles.")
print(pond[VARIABLES].describe().loc[["min", "mean", "max"]].round(3).to_string())


## Checks


In [ ]:
assert len(VARIABLES) == 9
assert pond[VARIABLES].notna().all().all()
assert pond.groupby("cycle_id")["day"].nunique().eq(CYCLE_DAYS).all()
assert pond[["farm_id", "cycle_id", "day"]].duplicated().sum() == 0
assert pond["pathogen_pressure"].between(0, 1).all()
print("PASS — nine-variable schema, completeness, uniqueness, cycle length, and bounds.")


## Takeaways

This notebook preserves multivariate pond dependence. Its output is a development dataset, not proof that simulated marginals represent every Indonesian pond. Run notebook 02 next.
